# Lab 5 - Event Producer

In this notebook I create fake Windows-style security events and send them to Kafka. This is the first part of the pipeline.

## What This Notebook Does

- Creates simple process and login events.
- Sends every event to the Kafka topic `raw-events`.
- Adds tracing spans so the producer can be seen in Jaeger.

In [1]:
%pip install -q kafka-python opentelemetry-sdk opentelemetry-exporter-otlp

Note: you may need to restart the kernel to use updated packages.


In [2]:
import json
import random
import time
import uuid
from datetime import datetime, timezone

from kafka import KafkaProducer

from opentelemetry import trace
from opentelemetry.exporter.otlp.proto.grpc.trace_exporter import OTLPSpanExporter
from opentelemetry.sdk.resources import Resource
from opentelemetry.sdk.trace import TracerProvider
from opentelemetry.sdk.trace.export import BatchSpanProcessor

KAFKA_BOOTSTRAP_SERVERS = "kafka:9092"
KAFKA_TOPIC = "raw-events"
OTLP_ENDPOINT = "jaeger:4317"
SERVICE_NAME = "lab5-producer"

resource = Resource.create({"service.name": SERVICE_NAME})
trace.set_tracer_provider(TracerProvider(resource=resource))
trace.get_tracer_provider().add_span_processor(
    BatchSpanProcessor(OTLPSpanExporter(endpoint=OTLP_ENDPOINT, insecure=True))
)
tracer = trace.get_tracer(__name__)

producer = KafkaProducer(
    bootstrap_servers=KAFKA_BOOTSTRAP_SERVERS,
    value_serializer=lambda value: json.dumps(value).encode("utf-8"),
)

print("Producer connected to Kafka")

Producer connected to Kafka


In [3]:
users = ["daniel", "alice", "bob", "service_account"]
hosts = ["WIN-01", "WIN-02", "LAPTOP-7", "SERVER-1"]
ips = ["10.0.0.10", "10.0.0.15", "10.0.0.25", "192.168.1.50"]

def make_event():
    event_type = random.choice(["process_start", "user_login"])
    event = {
        "event_id": str(uuid.uuid4()),
        "timestamp": datetime.now(timezone.utc).isoformat(),
        "user": random.choice(users),
        "host": random.choice(hosts),
        "source_ip": random.choice(ips),
        "event_type": event_type,
    }

    if event_type == "process_start":
        event["command_line"] = random.choice([
            "powershell.exe -EncodedCommand SQBFAFgA",
            "cmd.exe /c whoami",
            "notepad.exe",
        ])
    else:
        event["logon_type"] = random.choice(["success", "failure"])

    return event

In [4]:
EVENT_COUNT = 30

for _ in range(EVENT_COUNT):
    event = make_event()

    with tracer.start_as_current_span("produce_event") as span:
        span.set_attribute("event.id", event["event_id"])
        span.set_attribute("event.type", event["event_type"])
        span.set_attribute("kafka.topic", KAFKA_TOPIC)
        producer.send(KAFKA_TOPIC, event)

    print(f"Sent {event['event_id']} - {event['event_type']}")
    time.sleep(0.2)

producer.flush()
print(f"Finished sending {EVENT_COUNT} events")

Sent 1c6b5fe0-8cc4-419c-a4bc-a358dfbc0eca - user_login
Sent d4c9a61f-92cf-4e5a-b270-42f3dabc6a57 - user_login
Sent 1098fda5-504a-4788-a17b-342f4f2dec6b - user_login
Sent def09e4f-6808-413d-8dce-6b2c0198e365 - user_login
Sent 38a3278c-e5d5-4ae1-944e-5c7066fe1e3c - user_login
Sent ed3bfadc-8124-40ed-b3b6-60a186df7cc0 - process_start
Sent 0cb8cedc-8041-4b92-8f26-8d08d549bb58 - user_login
Sent b52520d6-7c38-43ed-930e-3145b1cc36e9 - user_login
Sent 585a2e4a-e3d5-4593-98c3-8c8ac8404242 - process_start
Sent 5272c23e-7586-4951-8de1-14a22dfe300f - user_login
Sent 67deb494-9ef1-4f6a-b847-8b8973365b33 - user_login
Sent 61d86f44-a7e1-4798-9376-73a8082ca44f - process_start
Sent 942c7504-b658-41fe-82de-8cfc5f67a712 - user_login
Sent 4f018773-36b8-4734-9a8e-fb817bd68338 - user_login
Sent cded657e-cfc2-4a2b-bdc9-6bdc6be78415 - process_start
Sent f927095e-500b-45bd-b3e7-cd1ed9baeaba - user_login
Sent bd4b02aa-7ca0-452d-bb26-c329f8dbb046 - process_start
Sent 75cdc492-70c7-4758-8af1-9082c203e737 - user_l